[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rusher3265/DFT/blob/main/Tantalum_DFT_Colab.ipynb)

# Tantalum DFT and superconductivity on Google Colab

A complete source package is embedded in this notebook. It retrieves a real Ta pseudopotential online and runs Quantum ESPRESSO 7.5 / EPW 6.0. The model is **bulk nonmagnetic alpha-Ta**, bcc, space group 229, at ambient pressure.

**First run:** choose a Python 3 CPU runtime, save a copy of this notebook to Drive, and run sections 1–5 in order. Enable the later stages only when ready. The initial settings are a **pilot**, not converged material predictions. A band gap in ordinary DFT is not a superconducting gap.

No API key is needed for the default workflow. Optional Materials Project import requires your own key. All heavy calculation I/O stays under `/content/ta_run`; completed stages can be copied to Google Drive.

## 1. Install the included Python workflow

This cell verifies and unpacks the embedded source ZIP and installs the Python dependencies. Keep `tantalum_source.zip` as a source backup. Section 3 copies it to Drive when Drive is enabled.

In [ ]:
import base64, hashlib, io, pathlib, subprocess, sys, zipfile
SOURCE = pathlib.Path("/content/tantalum_source")
SOURCE.mkdir(parents=True, exist_ok=True)
_payload = base64.b64decode('...')
assert hashlib.sha256(_payload).hexdigest() == '8d5e788ac3b5bee9741bd4be7d9c52d8dbf4a81764dec91590d252b04a35545a'
SOURCE_ZIP = pathlib.Path("/content/tantalum_source.zip")
SOURCE_ZIP.write_bytes(_payload)
with zipfile.ZipFile(io.BytesIO(_payload)) as _zip:
    for _entry in _zip.infolist():
        _relative = pathlib.PurePosixPath(_entry.filename)
        if _relative.is_absolute() or ".." in _relative.parts:
            raise ValueError("Unsafe embedded source")
        _target = SOURCE.joinpath(*_relative.parts)
        _target.parent.mkdir(parents=True, exist_ok=True)
        _target.write_bytes(_zip.read(_entry))
subprocess.run([sys.executable,"-m","pip","install","--quiet","--disable-pip-version-check",str(SOURCE)],check=True)
print("Installed source:",SOURCE)

## 2. Install the calculation engines

This installs the pinned conda-forge CPU build `qe=7.5=h19104ac_2`, containing QE 7.5 and EPW 6.0, using a checksum-verified micromamba installer. Allow several minutes and several hundred MB for dependencies. Selecting a GPU will not accelerate this CPU build.

In [ ]:
from ta_dft.setup import install_qe
QE_BIN = install_qe(prefix="/content/qe75",tools="/content/ta_tools")
print("Engines:",QE_BIN)

## 3. Open or restore a persistent project

Leave `USE_DRIVE=True` for checkpoints. Mounting Drive prompts you to grant access. Active calculations use local Colab storage; Drive receives verified archives after completed stages. Old archives are retained, so check your Drive quota.

After a runtime reset, rerun sections 1–3 with the same `ROOT` and `DRIVE_NAME`. Restoration occurs only into an empty local project. Choose different names for a separate study. Keep the same local path for exact QE restart compatibility.

In [ ]:
from pathlib import Path
import json,os,shutil
from ta_dft import Project
from ta_dft.project import runtime_info
os.environ.setdefault("MPLCONFIGDIR","/content/ta_tools/matplotlib")

USE_DRIVE = True
ROOT = Path("/content/ta_run")
DRIVE_NAME = "Tantalum_DFT_alpha"
CHECKPOINTS = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_FOLDER = Path("/content/drive/MyDrive")/DRIVE_NAME
    CHECKPOINTS = DRIVE_FOLDER/"checkpoints"
    DRIVE_FOLDER.mkdir(parents=True,exist_ok=True)
    shutil.copy2(SOURCE_ZIP,DRIVE_FOLDER/SOURCE_ZIP.name)

empty = not ROOT.exists() or not any(ROOT.iterdir())
if empty and CHECKPOINTS and (CHECKPOINTS/"latest.json").exists():
    project = Project.restore(CHECKPOINTS,root=ROOT)
    print("Restored latest checksum-verified checkpoint")
else:
    project = Project(ROOT,profile="pilot")
if project.config["qe_bin"] != QE_BIN:
    project.configure(qe_bin=QE_BIN)

def run_stage(function,*args,**kwargs):
    result = function(*args,**kwargs)
    if CHECKPOINTS:
        print("Checkpoint:",project.checkpoint(CHECKPOINTS))
    return result

print(json.dumps(runtime_info(),indent=2))
print(json.dumps(project.config,indent=2))

## 4. Retrieve inputs

The starting structure is reconstructed from NIST JARVIS JVASP-1014 metadata: one primitive bcc atom, conventional lattice parameter 3.312 Angstrom. The downloader verifies the live page when accessible and clearly labels use of the bundled metadata snapshot otherwise. This is not a downloaded CIF. NIST's original functional was optB88-vdW, so re-optimize with the local PBE setup.

The PseudoDojo Ta13 PBE norm-conserving potential is downloaded online and checked for element, functional, semicore count, SOC and file hash. For a separate SOC project, set `soc=True` before fetching. A matching fully relativistic potential is then used throughout the generated calculation chain.

In [ ]:
# For a separate SOC project only:
# project.configure(soc=True)
inputs = run_stage(project.fetch,source="nist",allow_cached_reference=True)
print(json.dumps(inputs,indent=2))

## 5. Run the first real DFT calculation

The pilot uses 80/320 Ry cutoffs, a 6³ electronic mesh, 16 bands, and 0.02 Ry metallic smearing. It tests the calculation pipeline. It does not establish converged physics. The cell prints log locations; repeating it reuses validated completed work.

In [ ]:
scf = run_stage(project.scf)
print({k:scf.get(k) for k in ["energy_ev","fermi_ev","elapsed_seconds","reused","output"]})

## 6. Check numerical convergence

Set the flag to True when ready. Review energy, pressure and Fermi-level stability. Increase meshes toward 16³, 20³ and 24³ as resources permit, then recheck the DOS, phonons and coupling. These are separate one-dimensional sweeps; convergence of total energy does not certify superconductivity calculations. Smearing is not the physical sample temperature.

In [ ]:
RUN_CONVERGENCE = False
if RUN_CONVERGENCE:
    study = run_stage(project.converge,cutoffs_ry=[80,100,120],
        kmeshes=[6,8,12],smearings_ry=[0.03,0.02,0.01])
    print(json.dumps(study,indent=2))
else:
    print("Set RUN_CONVERGENCE=True to calculate the sweeps.")

# Adopt reviewed settings explicitly, for example:
# project.configure(ecutwfc_ry=100.,ecutrho_ry=400.,kmesh=[16,16,16])

## 7. Optimize the lattice

After choosing numerical settings, fit an energy-versus-volume curve. This returns the lattice constant and bulk modulus. Adoption is explicit. The calculation is static-lattice electronic DFT, without zero-point or thermal expansion corrections.

In [ ]:
RUN_EOS = False
ADOPT_EQUILIBRIUM = True
if RUN_EOS:
    eos = run_stage(project.equation_of_state)
    print(json.dumps(eos,indent=2))
    if ADOPT_EQUILIBRIUM:
        project.configure(a_angstrom=eos["a_angstrom"])
        if CHECKPOINTS: print(project.checkpoint(CHECKPOINTS))
else:
    print("Set RUN_EOS=True after reviewing convergence.")

## 8. Electronic bands and density of states

Generate bands, total DOS and orbital projections using separate copies of the ground state for each branch. These are normal-state electronic properties. Compare matched scalar and SOC studies using separate project folders.

In [ ]:
RUN_ELECTRONIC = False
if RUN_ELECTRONIC:
    electronic = run_stage(project.electronic)
    print(json.dumps(electronic,indent=2))
else:
    print("Set RUN_ELECTRONIC=True for bands, DOS and projected DOS.")

## 9. Phonons in restartable q-point chunks

Set the flag to True and rerun this cell until `status` is `complete`. Each call finishes at most one unfinished irreducible q-point and checkpoints the project. Interrupted unfinished q-points are recalculated; completed points are retained. A complete grid produces phonon dispersion and DOS.

The 2³ q-grid is a pilot. Test 4³ and 6³ for a serious study. Inspect imaginary modes; significant imaginary frequencies block the superconductivity stage. The stability check covers the sampled path, not every point in the Brillouin zone.

In [ ]:
RUN_PHONON_CHUNK = False
if RUN_PHONON_CHUNK:
    phonons = run_stage(project.phonons,max_qpoints=1)
    print(json.dumps(phonons,indent=2))
else:
    print("Set RUN_PHONON_CHUNK=True; repeat the cell until complete.")

## 10. EPW preparation and independent interpolation validation

EPW uses an 8³ coarse electronic grid and nine Ta s/p/d Wannier functions by default. The occupied 5s/5p semicore states remain in DFT but are excluded from the Wannier subspace. SOC doubles the subspace and exclusions. Preparation can take several minutes and restarts that stage if interrupted; DFPT checkpoints remain available.

The validation compares direct DFT and interpolated bands near the Fermi energy along a path and at independent points, plus EPW versus matdyn phonons. A failed validation blocks dense EPW calculations. Refine the coarse mesh or orbital representation when needed; do not loosen the tolerance to force a result. EPW-only settings can change while reusing physical DFPT data.

In [ ]:
RUN_EPW_PREPARATION = False
if RUN_EPW_PREPARATION:
    prepared = run_stage(project.prepare_epw)
    validation = run_stage(project.validate_wannier,window_ev=1.,tolerance_ev=.05,
        points_per_segment=6,random_points=8)
    print(json.dumps(validation,indent=2))
else:
    print("Set RUN_EPW_PREPARATION=True after complete stable phonons.")

# If validation fails, inspect it and refine the coarse model, for example:
# project.configure(epw_coarse_kmesh=[12,12,12])
# Then repeat preparation and validation.

## 11. Electron–phonon spectrum and transition-temperature sensitivity

The fine 12³ electronic / 6³ phonon meshes below are also pilot settings. Increase them independently and vary integration broadening to check lambda, logarithmic frequency and Tc. The analysis uses the full Allen–Dynes expression with strength/shape factors at mu*=0.10, 0.13 and 0.15. Model sensitivity is separate from numerical convergence.

Bulk-Ta's experimental transition near 4.48 K is a comparison benchmark, not a parameter-fitting target. Ordinary DFT alone does not prove superconductivity.

In [ ]:
RUN_EPW = False
if RUN_EPW:
    epw = run_stage(project.epw,fine_kmesh=[12]*3,fine_qmesh=[6]*3,mu_star=.13)
    analysis = run_stage(project.analyze_a2f,epw["a2f_file"],frequency_unit="meV",
        mu_stars=[.10,.13,.15])
    print(json.dumps(analysis,indent=2))
else:
    print("Set RUN_EPW=True after the interpolation gate passes.")

## 12. Optional imaginary-axis gap calculation

Enable only after the spectrum stage works. This solves isotropic Migdal–Eliashberg equations at the selected temperatures. Verify convergence and extend the temperature range to bracket a transition. The reported lowest-Matsubara-frequency pairing function is a proxy, not a real-axis tunneling gap. Recheck the Matsubara cutoff and numerical meshes.

In [ ]:
RUN_GAP = False
if RUN_GAP:
    gap = run_stage(project.epw,fine_kmesh=[12]*3,fine_qmesh=[6]*3,
        temperatures=[1.,2.,3.,4.,5.,6.,8.,10.],mu_star=.13,solve_eliashberg=True)
    print(json.dumps(gap,indent=2))
else:
    print("Optional: set RUN_GAP=True after inspecting the spectrum.")

## 13. Analyze a spectrum calculated elsewhere

Optional: upload an alpha2F file. Specify its frequency/energy unit. EPW's alpha2F amplitude is dimensionless in the supported convention. For several broadening columns, explicitly select a spectral column; cumulative lambda and phonon DOS columns are not alpha2F.

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# spectrum_path = next(iter(uploaded))
# analysis = project.analyze_a2f(spectrum_path,frequency_unit="meV",columns=(0,1))
# print(json.dumps(analysis,indent=2))

## 14. Make plots and download results

This works at any stage and reports missing calculations honestly. It plots existing bands, DOS, phonons, convergence data, coupling spectra and temperature sensitivity. The portable report contains HTML, JSON and figure assets. A report is not a calculation restart backup: keep your full Drive checkpoints too.

In [ ]:
from IPython.display import display,HTML
import mimetypes
report = run_stage(project.report)
print(json.dumps(report,indent=2))
report_path = Path(report["html"])
preview = report_path.read_text()
for picture in report_path.parent.rglob("*"):
    if picture.suffix.lower() in (".png",".svg"):
        relative = picture.relative_to(report_path.parent).as_posix()
        data_url = "data:"+(mimetypes.guess_type(picture.name)[0] or "image/png")+";base64,"+base64.b64encode(picture.read_bytes()).decode()
        preview = preview.replace("src='"+relative+"'","src='"+data_url+"'")
display(HTML(preview))
print(json.dumps(project.status(),indent=2))

In [ ]:
from google.colab import files
files.download(report["archive"])
# Optional source backup to your computer:
# files.download(str(SOURCE_ZIP))

## Troubleshooting and references

- Reconnect: rerun sections 1–3 with the same names, then repeat the interrupted stage. Keep all .save, .phsave, wavefunction and dvscf files.
- A checkpoint failure does not erase the calculation. Check Drive space/mount, stop active calculations, and retry `project.checkpoint(CHECKPOINTS)`.
- An engine time limit or failed convergence is not marked as completed. Inspect the printed log path before changing settings. Most scientific-setting changes create a new isolated calculation tree.
- Colab resources vary. Benchmark before using dense production meshes; a completed pilot is not evidence that a full convergence study fits one session.
- This code models bulk alpha-Ta. Beta-Ta, films, disorder, oxides and device losses need separate models.

Sources: [NIST Ta metadata](https://www.ctcms.nist.gov/~knc6/jsmol/JVASP-1014.html), [PseudoDojo](https://www.pseudo-dojo.org/), [QE/EPW setup](https://docs.epw-code.org/tutorials/setup/index.html), [EPW interpolation](https://docs.epw-code.org/tutorials/tutorial_01/index.html), [EPW superconductivity](https://docs.epw-code.org/tutorials/tutorial_04/index.html), [Colab limits](https://research.google.com/colaboratory/faq.html), [bulk Ta experiment](https://doi.org/10.1103/PhysRev.119.1578).